In [28]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt
import plotly.graph_objects as go

In [29]:
cp_scen_nm = "Current-Policy"
ep_scen_nm = "Enhanced-Ambition"

In [30]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [31]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [32]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250712"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [33]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [34]:
scenarios = [cp_scen_nm, ep_scen_nm]

In [35]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [36]:
q = queries[84]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'
dfCO2

building service output by tech


,Units,scenario,region,sector,subsector,technology,Year,value,GHG
0,EJ,Current-Policy,South Korea,comm cooling,electricity,EE_electricity,2020,0.000014,CO2
1,EJ,Current-Policy,South Korea,comm cooling,electricity,EE_electricity,2025,0.000148,CO2
2,EJ,Current-Policy,South Korea,comm cooling,electricity,EE_electricity,2030,0.000274,CO2
3,EJ,Current-Policy,South Korea,comm cooling,electricity,EE_electricity,2035,0.000405,CO2
4,EJ,Current-Policy,South Korea,comm cooling,electricity,electricity,1990,0.007107,CO2
...,...,...,...,...,...,...,...,...,...
2460,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids,refined liquids,2015,0.004098,CO2
2461,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids,refined liquids,2020,0.003326,CO2
2462,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids,refined liquids,2025,0.003398,CO2
2463,EJ,Enhanced-Ambition,South Korea,resid others modern_d9,refined liquids,refined liquids,2030,0.002199,CO2


In [37]:
dfCO2['technology'].unique()

array(['EE_electricity', 'electricity', 'EE_gas', 'gas', 'biomass',
       'hydrogen', 'EE_refined liquids', 'refined liquids', 'coal',
       'traditional biomass'], dtype=object)

In [38]:
dfCO2[(dfCO2['sector'] == 'comm others')]

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
71,EJ,Current-Policy,South Korea,comm others,biomass,biomass,1990,0.000053,CO2
72,EJ,Current-Policy,South Korea,comm others,biomass,biomass,2005,0.000301,CO2
73,EJ,Current-Policy,South Korea,comm others,biomass,biomass,2010,0.000589,CO2
74,EJ,Current-Policy,South Korea,comm others,biomass,biomass,2015,0.001680,CO2
75,EJ,Current-Policy,South Korea,comm others,biomass,biomass,2020,0.001660,CO2
...,...,...,...,...,...,...,...,...,...
1354,EJ,Enhanced-Ambition,South Korea,comm others,refined liquids,refined liquids,2015,0.030854,CO2
1355,EJ,Enhanced-Ambition,South Korea,comm others,refined liquids,refined liquids,2020,0.027869,CO2
1356,EJ,Enhanced-Ambition,South Korea,comm others,refined liquids,refined liquids,2025,0.030082,CO2
1357,EJ,Enhanced-Ambition,South Korea,comm others,refined liquids,refined liquids,2030,0.018290,CO2


In [39]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['GHG'] = 'CO2'
q = queries[270]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfGHG = pd.concat([dfCO2, dfNonCO2], axis=0)
dfGHG['emiss(MT)'] = dfGHG.apply(convert_to_mt, axis=1)
dfGHG['gwpAr5'] = dfGHG['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHG['MTCO2eq'] = dfGHG['emiss(MT)'] * dfGHG['gwpAr5']
dfGHG

CO2 emissions by sector (no bio) (excluding resource production)
nonCO2 emissions by sector (excluding resource production)


,Units,scenario,region,sector,Year,value,GHG,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002832,CO2,0.010385,1.0,0.010385
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.011653,CO2,0.042728,1.0,0.042728
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.004342,CO2,0.015919,1.0,0.015919
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.008077,CO2,0.029615,1.0,0.029615
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002853,CO2,0.010460,1.0,0.010460
...,...,...,...,...,...,...,...,...,...,...
15510,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2015,0.000206,SO2_2,0.000206,NaN,NaN
15511,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2020,0.000210,SO2_2,0.000210,NaN,NaN
15512,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2025,0.000206,SO2_2,0.000206,NaN,NaN
15513,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,2030,0.000204,SO2_2,0.000204,NaN,NaN


In [40]:
dfGHG['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal_d1',
       'resid heating coal_d10', 'resid heating coal_d2',
       'resid heating coal_d3', 'resid heating coal_d4',
       'resid heating coal_d5', 'resid heating coal_d6',
       'resid he

In [41]:
dfHist = pd.read_excel("./extdata/gir2025.xlsx", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ].reset_index()
dfHist

분야·부문/연도,index,총배출량,순배출량,에너지,A. 연료연소,1. 에너지산업,a. 공공전기 및 열 생산,b. 석유정제,c. 고체연료 제조 및 기타 에너지 산업,2. 제조업 및 건설업,...,1. 퇴비화,2. 바이오가스시설에서의 혐기성 소화,C. 폐기물소각 및 노천소각,1. 폐기물소각,2. 노천소각,D. 하폐수처리,1. 하수처리,2. 폐수처리,3. 기타,E. 기타
0,1990,310578.34,271615.09,234475.75,223481.61,42026.93,37174.66,4193.73,658.54,72169.01,...,0.00,0.00,560.85,560.85,0.0,729.52,631.05,98.48,0.0,0.0
1,1991,341241.27,306225.60,257328.69,246864.64,49945.73,43773.82,4978.80,1193.10,85306.81,...,0.00,0.00,746.36,746.36,0.0,911.36,793.36,118.00,0.0,0.0
2,1992,368809.57,334622.72,273752.07,264236.12,59044.99,51282.99,6097.80,1664.21,88857.58,...,0.00,0.00,879.08,879.08,0.0,1070.06,919.56,150.50,0.0,0.0
3,1993,406802.15,374169.23,303458.93,294738.32,66967.19,58078.04,6787.82,2101.33,97533.10,...,0.00,0.00,1083.73,1083.73,0.0,889.59,715.65,173.94,0.0,0.0
4,1994,432769.28,397753.22,323158.10,315376.07,81318.26,71719.31,7046.10,2552.86,102910.98,...,4.26,0.00,1547.93,1547.93,0.0,1513.83,1277.13,236.71,0.0,0.0
5,1995,464497.63,431236.45,347962.44,340964.77,91952.43,79555.15,9291.77,3105.51,104629.69,...,0.51,0.00,2212.85,2212.85,0.0,1520.26,1238.24,282.02,0.0,0.0
6,1996,501079.14,464394.65,381519.41,374951.00,115575.06,96643.67,10525.32,8406.07,106236.46,...,1.57,0.00,1607.51,1607.51,0.0,1561.24,1406.53,154.71,0.0,0.0
7,1997,526063.93,484380.56,398735.84,392509.74,124958.96,104573.97,13160.26,7224.74,107331.55,...,4.68,0.00,2014.38,2014.38,0.0,1381.70,1231.28,150.42,0.0,0.0
8,1998,460219.69,410168.46,340200.65,334224.21,112389.64,92499.17,13398.10,6492.37,97052.38,...,4.50,0.00,1901.37,1901.37,0.0,1631.16,1504.14,127.02,0.0,0.0
9,1999,500569.62,442315.49,372901.35,367093.36,122425.73,101887.04,13874.42,6664.27,104531.80,...,16.13,0.00,3245.30,3245.30,0.0,1728.48,1573.45,155.02,0.0,0.0


In [42]:
dfHistCO2 = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="CO2", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistCO2

분야·부문/연도,총배출량,순배출량,에너지,A. 연료연소,1. 에너지산업,a. 공공전기 및 열 생산,b. 석유정제,c. 고체연료 제조 및 기타 에너지 산업,2. 제조업 및 건설업,a. 철강,...,1. 퇴비화,2. 바이오가스시설에서의 혐기성 소화,C. 폐기물소각 및 노천소각,1. 폐기물소각,2. 노천소각,D. 하폐수처리,1. 하수처리,2. 폐수처리,3. 기타,E. 기타
1990,256216.01,216831.36,219131.09,218944.11,41885.67,37044.76,4183.18,657.73,71922.38,27806.85,...,0.0,0.0,530.01,530.01,0.0,0.0,0.0,0.0,0.0,0.0
1991,286190.99,250790.40,242917.02,242753.45,49781.26,43623.51,4965.94,1191.81,85016.98,35147.82,...,0.0,0.0,705.28,705.28,0.0,0.0,0.0,0.0,0.0,0.0
1992,310353.52,275791.86,260584.43,260454.36,58854.01,51109.72,6081.84,1662.45,88560.19,37175.68,...,0.0,0.0,828.76,828.76,0.0,0.0,0.0,0.0,0.0,0.0
1993,347747.49,314771.79,291228.60,291125.90,66746.03,57876.59,6770.02,2099.41,97206.23,41547.99,...,0.0,0.0,1022.29,1022.29,0.0,0.0,0.0,0.0,0.0,0.0
1994,372197.71,336861.99,312113.66,312032.68,81046.89,71469.05,7027.56,2550.27,102557.48,41310.10,...,0.0,0.0,1476.73,1476.73,0.0,0.0,0.0,0.0,0.0,0.0
1995,401672.09,368091.79,337646.34,337583.94,91650.47,79276.27,9271.65,3102.56,104272.86,42942.78,...,0.0,0.0,2113.79,2113.79,0.0,0.0,0.0,0.0,0.0,0.0
1996,437066.43,400046.07,371494.17,371440.03,115179.69,96299.15,10501.91,8378.64,105882.54,43477.75,...,0.0,0.0,1508.67,1508.67,0.0,0.0,0.0,0.0,0.0,0.0
1997,457947.28,415925.84,388838.92,388789.45,124533.55,104197.73,13134.09,7201.73,106961.91,41549.44,...,0.0,0.0,1904.74,1904.74,0.0,0.0,0.0,0.0,0.0,0.0
1998,395004.07,344595.47,331253.17,331205.32,111984.27,92143.63,13371.07,6469.57,96730.74,42478.94,...,0.0,0.0,1797.16,1797.16,0.0,0.0,0.0,0.0,0.0,0.0
1999,430541.96,371940.93,363860.10,363813.91,121984.10,101496.92,13845.90,6641.28,104200.76,44929.72,...,0.0,0.0,3118.80,3118.80,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
dfHistCO2 = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="CO2", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistCH4 = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="CH4", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistN2O = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="N2O", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistSF6 = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="SF6", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistHFC = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="HFCs", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistPFC = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="PFCs", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
# dfHistNF3 = pd.read_excel("./extdata/gir2025.xlsx", sheet_name="NF3", skiprows=3, index_col=1, skipfooter=9).transpose().iloc[1:, ]
dfHistGas = pd.DataFrame({
    "CO2": dfHistCO2['총배출량'],
    "CH4": dfHistCH4['총배출량'],
    "N2O": dfHistN2O['총배출량'],
    "F-Gases": dfHistSF6['총배출량'] + dfHistHFC['총배출량'] + dfHistPFC['총배출량'],# + dfHistNF3['총배출량'],
})
dfHistGas.reset_index(inplace=True, names=['Year'])
# Compute cumulative stack
dfHistGas['CO2_stack'] = dfHistGas['CO2'] / 1000
dfHistGas['CH4_stack'] = (dfHistGas['CO2'] + dfHistGas['CH4']) / 1000
dfHistGas['N2O_stack'] = (dfHistGas['CO2'] + dfHistGas['CH4'] + dfHistGas['N2O']) / 1000
dfHistGas['FGas_stack'] = (dfHistGas['CO2'] + dfHistGas['CH4'] + dfHistGas['N2O'] + dfHistGas['F-Gases']) / 1000
dfHistGas

,Year,CO2,CH4,N2O,F-Gases,CO2_stack,CH4_stack,N2O_stack,FGas_stack
0,1990,256216.01,46817.79,6333.87,1210.67,256.21601,303.03380,309.36767,310.57834
1,1991,286190.99,47134.11,6744.65,1171.51,286.19099,333.32510,340.06975,341.24126
2,1992,310353.52,47269.47,8863.54,2323.05,310.35352,357.62299,366.48653,368.80958
3,1993,347747.49,47130.25,9309.21,2615.20,347.74749,394.87774,404.18695,406.80215
4,1994,372197.71,47350.74,10044.89,3175.94,372.19771,419.54845,429.59334,432.76928
5,1995,401672.09,47099.73,10901.91,4823.90,401.67209,448.77182,459.67373,464.49763
6,1996,437066.43,47050.01,11866.50,5096.20,437.06643,484.11644,495.98294,501.07914
7,1997,457947.28,47028.11,12784.21,8304.33,457.94728,504.97539,517.75960,526.06393
8,1998,395004.07,45635.72,13009.24,6570.65,395.00407,440.63979,453.64903,460.21968
9,1999,430541.96,44850.68,13910.74,11266.25,430.54196,475.39264,489.30338,500.56963


In [44]:
dfOut = dfGHG[~(dfGHG['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum().reset_index()
dfOut.rename(columns={'MTCO2eq': 'value'}, inplace=True)
dfOut['Units'] = 'MTCO2eq'

In [45]:
dfOut = dfOut.set_index(['scenario', 'Year'])
offset = 0 #624.5255 - 621.1791
dfOut.loc[(cp_scen_nm, 2005), 'value'] -= 57.5
dfOut.loc[(cp_scen_nm, 2010), 'value'] -= 57.3
dfOut.loc[(cp_scen_nm, 2015), 'value'] -= 47.8
dfOut.loc[(cp_scen_nm, 2020), 'value'] -= (38.8 + offset)
dfOut.loc[(cp_scen_nm, 2025), 'value'] -= (30.4 + offset)
dfOut.loc[(cp_scen_nm, 2030), 'value'] -= (21.8 + offset)
dfOut.loc[(cp_scen_nm, 2035), 'value'] -= (13.2 + offset)
dfOut.loc[(ep_scen_nm, 2005), 'value'] -= 57.5
dfOut.loc[(ep_scen_nm, 2010), 'value'] -= 57.3
dfOut.loc[(ep_scen_nm, 2015), 'value'] -= 47.8
dfOut.loc[(ep_scen_nm, 2020), 'value'] -= 38.8
dfOut.loc[(ep_scen_nm, 2025), 'value'] -= 30.4
dfOut.loc[(ep_scen_nm, 2030), 'value'] -= 38.8
dfOut.loc[(ep_scen_nm, 2035), 'value'] -= 47.8
dfOut = dfOut.reset_index()

In [46]:
dfOut

,scenario,Year,value,Units
0,Current-Policy,1975,41.320031,MTCO2eq
1,Current-Policy,1990,274.711343,MTCO2eq
2,Current-Policy,2005,495.845816,MTCO2eq
3,Current-Policy,2010,597.361819,MTCO2eq
4,Current-Policy,2015,656.830290,MTCO2eq
5,Current-Policy,2020,638.416459,MTCO2eq
6,Current-Policy,2025,624.032674,MTCO2eq
7,Current-Policy,2030,567.341834,MTCO2eq
8,Current-Policy,2035,493.645347,MTCO2eq
9,Enhanced-Ambition,1975,41.320031,MTCO2eq


In [20]:
dfPower = dfGHG[(dfGHG['sector'].str.contains('elec'))]# | (dfGHG['sector'].isin(lstOtherPower))].copy()
dfOutPower = dfPower.groupby(['scenario', 'region', 'Year'])['MTCO2eq'].sum().reset_index()
dfOutPower.rename(columns={'MTCO2eq': 'value'}, inplace=True)
dfOutPower['Units'] = 'MTCO2eq'
dfOutPower

,scenario,region,Year,value,Units
0,Current-Policy,South Korea,1990,35.315064,MTCO2eq
1,Current-Policy,South Korea,2005,177.820212,MTCO2eq
2,Current-Policy,South Korea,2010,251.589396,MTCO2eq
3,Current-Policy,South Korea,2015,260.227190,MTCO2eq
4,Current-Policy,South Korea,2020,246.910617,MTCO2eq
5,Current-Policy,South Korea,2025,211.221179,MTCO2eq
6,Current-Policy,South Korea,2030,165.716890,MTCO2eq
7,Current-Policy,South Korea,2035,122.260760,MTCO2eq
8,Enhanced-Ambition,South Korea,1990,35.315064,MTCO2eq
9,Enhanced-Ambition,South Korea,2005,177.820212,MTCO2eq


In [51]:
fig = go.Figure()

# emiss_2018 = 727.6
emiss_2018 = 783.8
# emiss_2018_net = 742.31
x_center = 2040
width = 1
# offset = 4.9574  # BPESD upward shift

emiss_2035_cur = dfOut.set_index(['scenario', 'Year'])['value'].get((cp_scen_nm, 2035))
emiss_2035_enh = dfOut.set_index(['scenario', 'Year'])['value'].get((ep_scen_nm, 2035))

reduc_cur = (1 - (emiss_2035_cur / emiss_2018)) * 100
reduc_enh = (1 - (emiss_2035_enh / emiss_2018)) * 100

# === Legend Group: Gases ===
# fig.add_trace(go.Scatter(
#     x=[None], y=[None], mode='lines',
#     line=dict(color='rgba(0,0,0,0)'),
#     name='<b>Sector</b>', showlegend=True, hoverinfo='skip'
# ))

# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['순배출량'] / 1000,
#     mode='lines', name='Total GHG (incl. LULUCF)',
#     line=dict(color='black', width=1)
# ))


# fig.add_trace(go.Scatter(
#     x=dfHist['index'],
#     y=dfHist['a. 공공전기 및 열 생산'] / 1000,
#     mode='lines', name='Electricity',
#     line=dict(color='black', width=1, dash='dash')
# ))

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Gas</b>', showlegend=True, hoverinfo='skip'
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CO2_stack'],
    mode='lines', name='CO2', fill='tozeroy',
    line=dict(width=0.5, color='grey')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['CH4_stack'],
    mode='lines', name='CH4', fill='tonexty',
    line=dict(width=0.5, color='green')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['N2O_stack'],
    mode='lines', name='N2O', fill='tonexty',
    line=dict(width=0.5, color='purple')
))
fig.add_trace(go.Scatter(x=dfHistGas['Year'], y=dfHistGas['FGas_stack'],
    mode='lines', name='F-Gases', fill='tonexty',
    line=dict(width=0.5, color='yellow')
))

fig.add_trace(go.Scatter(
    x=[2030], y=[emiss_2018 * 0.6 + 37.5],
    mode='markers+text',
    marker=dict(color='#750D86', size=7, symbol='triangle-up'),
    text='2030 NDC', textposition="middle left",
    textfont=dict(size=15, color="#750D86"),
    showlegend=False
))

# === Static Reference Point ===
fig.add_trace(go.Scatter(
    x=[2018], y=[emiss_2018],
    mode='markers+text',
    marker=dict(color='black', size=7),
    text=f'2018: {emiss_2018} MtCO2e',
    textposition='middle right',
    textfont=dict(size=15),
    showlegend=False
))

# === Legend Group: Scenarios ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Scenario</b>', showlegend=True, hoverinfo='skip'
))

fig.add_trace(go.Scatter(
    x=dfHist['index'],
    y=dfHist['순배출량'] / 1000,
    mode='lines', name='Historical Data',
    line=dict(color='black', width=1)
))

fig.add_trace(go.Scatter(
    x=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
    y=dfOut[(dfOut['scenario'] == cp_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
    mode='lines', name='Current Policy',
    line=dict(color='#636EFA', width=1.5)
))

fig.add_trace(go.Scatter(
    x=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['Year'],
    y=dfOut[(dfOut['scenario'] == ep_scen_nm) & (dfOut['Year'] >= 2025)]['value'],
    mode='lines', name='Enhanced Ambition',
    line=dict(color='#00CC96', width=1.5)
))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == cp_scen_nm) & (dfOutPower['Year'] >= 2025)]['value'],
#     mode='lines', name='Current Policy',
#     line=dict(color='#636EFA', width=1.5, dash='dash'),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['Year'],
#     y=dfOutPower[(dfOutPower['scenario'] == ep_scen_nm) & (dfOutPower['Year'] >= 2005)]['value'],
#     mode='lines', name='Enhanced Ambition',
#     line=dict(color='#00CC96', width=1.5, dash='dash'),
#     showlegend=False
# ))

# === Axes & Layout ===
fig.update_layout(
    title="<b>Greenhouse gas emission pathways of South Korea</b>",
    title_font_size=28,
    title_x=0.43,
    legend_traceorder="normal",
    legend_title='',
    legend_font_size=15,
    plot_bgcolor='rgba(0,0,0,0)',
    width=1200,
    height=700,
    xaxis=dict(
        showgrid=False,
        title="Year", title_font_size=25,
        tickvals=list(range(1990, 2040, 5)),
        tickfont_size=15,
        range=[1987, 2045]
    ),
    yaxis=dict(
        showgrid=False,
        title="Emission (MtCO2e)", title_font_size=25,
        tickvals=list(range(0, 801, 100)),
        tickfont_size=15,
        range=[-30, 830]
    )
)

# === Reduction Boxes (overlay bars) ===
fig.add_trace(go.Scatter(
    x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
    y=[emiss_2035_cur, emiss_2035_cur, emiss_2035_enh, emiss_2035_enh],
    fill='toself', fillcolor='rgba(0,204,150,0.3)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
))
fig.add_trace(go.Scatter(
    x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
    y=[emiss_2018, emiss_2018, emiss_2035_cur, emiss_2035_cur],
    fill='toself', fillcolor='rgba(99,110,250,0.3)',
    line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
))

# === Reduction Labels ===
fig.add_annotation(
    x=x_center - 0.15, y=(emiss_2018 + emiss_2035_cur) / 2,
    text=f"<b>Current<br>Policy<br>-{reduc_cur:.1f}%</b>",
    showarrow=False, font=dict(size=14, color="#636EFA")
)
fig.add_annotation(
    x=x_center + 0.15, y=(emiss_2035_enh + emiss_2035_cur) / 2,
    text=f"<b>Enhanced<br>Ambition<br>-{reduc_enh:.1f}%</b>",
    showarrow=False, font=dict(size=14, color="#00CC96")
)

# === Reduction Boxes (overlay bars) ===
# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[119.0679, 119.0679, 40.657, 40.657],
#     fill='toself', fillcolor='rgba(0,204,150,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# fig.add_trace(go.Bar(
#     x=[x_center],
#     y=[278.84-40.657],
#     base=[40.657],  # for positioning
#     width=width,
#     marker=dict(
#         color='rgba(99,110,250,0.3)',
#         pattern=dict(
#             shape='\\',  # options: '/', '\\', 'x', '-', '|', '+', '.'
#             fillmode='overlay',
#             size=5,
#             solidity=0.2
#         )
#     ),
#     showlegend=False
# ))

# fig.add_trace(go.Scatter(
#     x=[x_center - width / 2, x_center + width / 2, x_center + width / 2, x_center - width / 2],
#     y=[278.84, 278.84, 119.0679, 119.0679],
#     fill='toself', fillcolor='rgba(99,110,250,0.3)',
#     line=dict(color='rgba(0,0,0,0)'), hoverinfo='skip', showlegend=False
# ))

# reduc_cur_power = 100 * (1 - 119.0679/278.84)
# reduc_enh_power = 100 * (1 - 40.657/278.84)

# # === Reduction Labels ===
# fig.add_annotation(
#     x=x_center - 0.15, y=(278.84 + 119.0679) / 2,
#     text=f"<b>-{reduc_cur_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#636EFA")
# )
# fig.add_annotation(
#     x=x_center + 0.15, y=(40.657 + 119.0679) / 2,
#     text=f"<b>-{reduc_enh_power:.1f}%</b>",
#     showarrow=False, font=dict(size=14, color="#00CC96")
# )


# === Grid Lines (manual dashed lines) ===
for x in range(1990, 2040, 5):
    fig.add_shape(type="line", x0=x, x1=x, y0=-30, y1=830,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
for y in range(0, 831, 100):
    fig.add_shape(type="line", x0=1987, x1=2037, y0=y, y1=y,
                  line=dict(color="LightGrey", width=1, dash="dash"),
                  layer='below')
fig.add_shape(type="line", x0=1987, x1=2037, y0=0, y1=0,
              line=dict(color="LightGrey", width=1, dash="dash"),
              layer='above')

# === Optional: Highlight window 2025–2035 ===
fig.add_vrect(x0=2025, x1=2035, fillcolor="LightBlue", opacity=0.1,
              layer="below", line_width=0)

# === Box Around Plot ===
fig.add_shape(
    type="rect", xref="paper", yref="paper",
    x0=0, x1=1, y0=0, y1=1,
    line=dict(color="black", width=1),
    layer="above"
)

fig.add_annotation(
    text="w/o International Offset",
    xref="paper", 
    yref="paper",
    x=0.5,
    y=1.08,
    showarrow=False,
    font=dict(size=21, color="grey"),
    align="center"
)
fig.write_image("./fig/emiss_tot.png", scale=1)
fig

In [23]:
plotly_get_chrome

NameError: name 'plotly_get_chrome' is not defined